# Практика 16. Кореляційний аналіз: діаграма розсіювання, коефіцієнти Пірсона і Спірмена, кореляційна матриця

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса

Мета: дослідити зв'язок температури з витратами на опалення, порівняти кореляції Пірсона й Спірмена та оцінити вплив аномального спостереження.

## Побудова набору даних варіанта

Місячні температури Одеси беруться з таблиці варіантів. Опалювальні дні та витрати генеруються за заданими формулами з фіксованим seed для відтворюваності.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

np.random.seed(3)
city = "Одеса"
monthly_temp = [-1, 0, 4, 10, 16, 20, 23, 22, 17, 11, 5, 1]
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

rows = []
for month, temp in zip(months, monthly_temp):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": city,
        "місяць": month,
        "температура": temp,
        "опалювальні_дні": heating_days,
        "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)
display(climate)

## Завдання 1. Діаграма розсіювання

На графіку температура місяця відкладена по осі X, витрати на опалення — по осі Y.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(climate["температура"], climate["витрати_на_опалення"])
ax.set_xlabel("Середньомісячна температура (°C)")
ax.set_ylabel("Витрати на опалення (умовні тис. грн)")
ax.set_title(f"Температура і витрати на опалення — {city}")
ax.grid(alpha=0.25)
plt.show()

На діаграмі очікується загалом спадний зв'язок: коли температура вища, опалювальних днів і витрат зазвичай менше. Форма може бути приблизно лінійною, але генераційний шум, округлення й обмеження витрат знизу нулем можуть створити розкид або точки, що відхиляються від загального тренду. Це синтетичні дані, тому їхня форма також залежить від випадкового seed.

## Завдання 2. Кореляції Пірсона і Спірмена

Пірсон оцінює лінійний зв'язок значень, а Спірмен — монотонний зв'язок через ранги. Для Пірсона також отримаємо 95% довірчий інтервал для генерального коефіцієнта кореляції.

In [ ]:
alpha = 0.05
temperature = climate["температура"]
heating_cost = climate["витрати_на_опалення"]
pearson_result = pearsonr(temperature, heating_cost)
spearman_result = spearmanr(temperature, heating_cost)
pearson_ci = pearson_result.confidence_interval(confidence_level=0.95)

print(f"Пірсон: r={pearson_result.statistic:.4f}, p={pearson_result.pvalue:.6f}")
print(f"95% ДІ для генеральної кореляції Пірсона: ({pearson_ci.low:.4f}, {pearson_ci.high:.4f})")
print(f"Спірмен: ρ={spearman_result.statistic:.4f}, p={spearman_result.pvalue:.6f}")
print("Пірсон: ", "відхиляємо H₀ про нульову кореляцію" if pearson_result.pvalue < alpha else "немає підстав відхилити H₀ про нульову кореляцію")
print("Спірмен:", "відхиляємо H₀ про нульову кореляцію" if spearman_result.pvalue < alpha else "немає підстав відхилити H₀ про нульову кореляцію")

Порівняйте отримані `r` і `ρ` та p-value: близькі коефіцієнти підтримують висновок, що спадний зв'язок близький до монотонного й приблизно лінійного; помітна різниця може свідчити про нелінійність або вплив крайніх значень. 95% довірчий інтервал описує невизначеність оцінки генеральної кореляції: за повторення процедури інтервали, побудовані таким самим способом, приблизно у 95% випадків накривали б справжнє значення кореляції.

## Завдання 3. Кореляційні матриці

Побудуємо матрицю Пірсона та рангову матрицю Спірмена для температури, кількості опалювальних днів і витрат на опалення.

In [ ]:
numeric_data = climate[["температура", "опалювальні_дні", "витрати_на_опалення"]]
pearson_matrix = numeric_data.corr(method="pearson")
spearman_matrix = numeric_data.corr(method="spearman")

print("Кореляційна матриця Пірсона:")
display(pearson_matrix.round(3))
print("Кореляційна матриця Спірмена:")
display(spearman_matrix.round(3))

pairs = [
    ("температура", "опалювальні_дні"),
    ("температура", "витрати_на_опалення"),
    ("опалювальні_дні", "витрати_на_опалення"),
]
strongest_pair = max(pairs, key=lambda pair: abs(pearson_matrix.loc[pair[0], pair[1]]))
weakest_pair = min(pairs, key=lambda pair: abs(pearson_matrix.loc[pair[0], pair[1]]))
print("Найсильніша пара за |r|:", strongest_pair, pearson_matrix.loc[strongest_pair[0], strongest_pair[1]])
print("Найслабша пара за |r|:", weakest_pair, pearson_matrix.loc[weakest_pair[0], weakest_pair[1]])

Кількість опалювальних днів формується безпосередньо з температури, а витрати — з кількості опалювальних днів із додатковим шумом. Тому зв'язки між цими змінними очікувані. Порівняйте матриці: Спірмен може краще відобразити монотонний порядок за наявності округлення чи нелінійності, а Пірсон найкраще підсумовує лінійний зв'язок.

## Завдання 4. Кореляція і причинність

У цьому синтетичному наборі температура безпосередньо входить до формули кількості опалювальних днів, а кількість днів входить до формули витрат. Тому зв'язок побудований за причинним напрямом у самій моделі генерації, а не лише спостережений як випадкова асоціація. Водночас кореляція сама по собі не доводить причинність у реальних даних: на витрати можуть впливати площа житла, тарифи, ізоляція будівель та інші конфаундери. На відміну від прикладу «морозиво й утоплення», де сезон/спека може бути спільною причиною обох явищ, тут зв'язок закладено безпосередньо формулами, хоча шум і спрощення моделі залишаються.

## Завдання 5. Вплив аномального спостереження

Змоделюємо аварію на котельні в одному місяці: витрати для найхолоднішого місяця збільшаться у п'ять разів. Порівняємо обидва коефіцієнти з початковими значеннями.

In [ ]:
anomalous_climate = climate.copy()
anomaly_index = anomalous_climate["температура"].idxmin()
original_cost = anomalous_climate.loc[anomaly_index, "витрати_на_опалення"]
anomalous_climate.loc[anomaly_index, "витрати_на_опалення"] = original_cost * 5

pearson_anomalous = pearsonr(
    anomalous_climate["температура"],
    anomalous_climate["витрати_на_опалення"],
)
spearman_anomalous = spearmanr(
    anomalous_climate["температура"],
    anomalous_climate["витрати_на_опалення"],
)

print(f"Аномальний рядок: {anomalous_climate.loc[anomaly_index, 'місяць']}, температура={anomalous_climate.loc[anomaly_index, 'температура']} °C")
print(f"Витрати до аварії: {original_cost:.1f}; після аварії: {anomalous_climate.loc[anomaly_index, 'витрати_на_опалення']:.1f}")
print(f"Пірсон без аномалії: r={pearson_result.statistic:.4f}; з аномалією: r={pearson_anomalous.statistic:.4f}; зміна={abs(pearson_anomalous.statistic - pearson_result.statistic):.4f}")
print(f"Спірмен без аномалії: ρ={spearman_result.statistic:.4f}; з аномалією: ρ={spearman_anomalous.statistic:.4f}; зміна={abs(spearman_anomalous.statistic - spearman_result.statistic):.4f}")

Порівняйте величини змін, надруковані вище. Пірсон використовує самі значення й чутливий до далекої точки, бо вона сильно впливає на середні та добутки відхилень. Спірмен використовує ранги, тому величезне числове значення витрат не збільшує його вплив безмежно; однак зміна рангу аномального спостереження все одно може змінити ρ.

## Контрольні питання

1. **Чому Пірсон нормує коваріацію добутком стандартних відхилень?** Ділення на `sₓsᵧ` прибирає одиниці вимірювання й переводить спільну мінливість у безрозмірну шкалу. За нерівністю Коші—Буняковського абсолютне значення стандартизованої коваріації не перевищує 1, тому `r ∈ [-1, 1]`.

2. **Чому Спірмен працює з рангами?** Ранги зберігають порядок значень, але зменшують залежність від відстаней між ними. Тому екстремальне значення зазвичай не може впливати пропорційно своїй числовій величині, що робить Спірмена стійкішим до викидів, хоча перестановка рангів усе одно має вплив.

3. **Приклад `r ≈ 0` за сильного зв'язку.** Якщо `Y = X²`, а значення `X` симетрично розташовані навколо нуля, зв'язок між `X` і `Y` є точним і сильним: знаючи `X`, ми знаємо `Y`. Але залежність U-подібна, а лінійні внески лівої та правої частин можуть взаємно компенсуватися, тож Пірсон може бути близьким до нуля.

4. **Чому кореляція не доводить вплив?** Кореляція описує спільну зміну змінних, але не встановлює напрям причинності й не виключає спільних причин, зворотного впливу або випадкового збігу. Для причинного висновку потрібен відповідний дизайн дослідження й контроль альтернативних пояснень.

## Підсумок

Для кліматичного набору Одеси побудовано діаграму розсіювання, обчислено кореляції Пірсона і Спірмена з довірчим інтервалом, порівняно матриці кореляцій і досліджено зміну коефіцієнтів після додавання аномального спостереження.